## Let Your Agent Write and Run Code: The Code Interpreter

In this notebook, we build an agent that can **write and execute Python code** on its own.  
We'll upload a week of coffee shop sales data and ask the agent to analyze it — calculating revenue, spotting trends, and generating charts, all without us writing a single line of analysis code ourselves.

### Step 1: Install Packages

We need the Azure AI Projects SDK, the OpenAI client, and a few helper libraries for authentication and configuration.

In [ ]:
# Install everything we need in a single command
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

### Step 2: Load Settings

Our project endpoint and model name are stored in a `.env` file.  
We load them here so the rest of the notebook can use them without hard-coding anything.

In [ ]:
# os lets us read environment variables from the system
import os

# load_dotenv reads your .env file and sets each line as an environment variable
from dotenv import load_dotenv

# DefaultAzureCredential automatically picks the best way to authenticate with Azure
from azure.identity import DefaultAzureCredential

# AIProjectClient is our main entry point for talking to Azure Foundry
from azure.ai.projects import AIProjectClient

# PromptAgentDefinition describes the agent -- its model, instructions, and tools
# CodeInterpreterTool gives the agent the ability to write and execute Python code
# CodeInterpreterToolAuto automatically manages the sandboxed code environment
from azure.ai.projects.models import PromptAgentDefinition, CodeInterpreterTool, CodeInterpreterToolAuto

# Load the .env file so os.getenv() can find our settings
load_dotenv()

# The URL that points to your Azure Foundry project
my_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")

# The name of the deployed language model the agent will use
my_model = os.getenv("MODEL_DEPLOYMENT_NAME")

### Step 3: Connect to Foundry

Create the project client that we will use for creating agents, uploading files, and more.

In [ ]:
# Initialize the Foundry client with our endpoint and Azure credentials
foundry_client = AIProjectClient(
    endpoint=my_endpoint,
    credential=DefaultAzureCredential(),
)

### Step 4: Create the Chat Client

The chat client is an OpenAI-compatible interface that lets us send messages and manage conversations.

In [ ]:
# This returns an OpenAI client that is already configured for our Foundry project
chat_client = foundry_client.get_openai_client()

### Step 5: Upload a Data File

The code interpreter needs data to work with.  
We upload a CSV file containing one week of coffee shop orders so the agent can read it, analyze it, and create visualizations from it.

In [ ]:
# Upload the CSV file to the Foundry service
# purpose="assistants" tells the service this file will be used by an agent/assistant
# We open the file in binary read mode ("rb") because the SDK expects raw bytes
uploaded_file = chat_client.files.create(
    purpose="assistants",
    file=open("./coffee_shop_sales.csv", "rb"),
)

print(f"Output Type: {output_item.type}")
output_item.code

# Print the file ID so we know the upload was successful
print(f"File uploaded successfully -- ID: {uploaded_file.id}")

### Step 6: Create an Agent with Code Interpreter Powers

This agent can write Python code, run it in a secure sandbox, and return the results.  
We attach the uploaded coffee shop sales file so the code interpreter has data to analyze.

In [ ]:
# Pick a descriptive name for this agent
coder_agent_name = "coffee-sales-analyst"

# Register the agent in Foundry with the code interpreter tool
coder_agent = foundry_client.agents.create_version(
    agent_name=coder_agent_name,
    definition=PromptAgentDefinition(
        # The language model that powers the agent's reasoning
        model=my_model,
        # Instructions that tell the agent what role it plays
        instructions="You are a data analyst for a coffee shop. You write and execute Python code to explore sales data, calculate revenue, identify best-selling drinks, and create clear visualizations. Always label your charts and explain your findings.",
        # Give the agent the code interpreter tool with our uploaded file attached
        tools=[
            CodeInterpreterTool(
                container=CodeInterpreterToolAuto(
                    # Pass the file ID so the code interpreter can access our CSV
                    file_ids=[uploaded_file.id]
                )
            )
        ],
    ),
)

# Confirm the agent was created
print(f"Agent ready -- ID: {coder_agent.id}, Name: {coder_agent.name}, Version: {coder_agent.version}")

### Step 7: Start a Conversation

Open a conversation thread so the agent can maintain context across multiple messages.

In [ ]:
# Create a fresh conversation for this session
chat_session = chat_client.conversations.create()

# Print the conversation ID
print(f"Conversation started -- ID: {chat_session.id}")

### Step 8: Ask the Agent to Analyze Your Data

We ask the agent to calculate total revenue per drink and create a chart.  
The agent will write Python code behind the scenes, run it, and return the result.

In [ ]:
# Send a data analysis request to the code interpreter agent
analysis_result = chat_client.responses.create(
    # Link this message to our conversation
    conversation=chat_session.id,
    # The task we want the agent to perform
    input="Calculate the total revenue (price * quantity) for each drink and create a horizontal bar chart showing total revenue by drink, sorted from highest to lowest.",
    # Point to our agent by name
    extra_body={
        "agent": {
            "name": coder_agent.name,
            "type": "agent_reference",
        }
    },
)

# Print the response ID to confirm the request completed
print(f"Analysis complete -- Response ID: {analysis_result.id}")

### Step 9: Download the Generated Chart

The code interpreter often generates files (like charts or CSVs).  
We look through the response annotations to find any generated files and download them.

In [ ]:
# These variables will hold the details of any file the agent generated
output_file_id = ""
output_filename = ""
output_container_id = ""

# The last item in the output list is usually the final message from the agent
final_output = analysis_result.output[-1]

# Check that it is a message (not a tool call or other type)
if final_output.type == "message":
    # Grab the last piece of content in that message
    text_block = final_output.content[-1]

    # Make sure it is a text block (which can contain annotations)
    if text_block.type == "output_text":
        # Annotations hold metadata about files the agent created
        if text_block.annotations:
            # Get the most recent annotation (the latest generated file)
            file_ref = text_block.annotations[-1]

            # container_file_citation means the agent saved a file in its sandbox
            if file_ref.type == "container_file_citation":
                output_file_id = file_ref.file_id
                output_filename = file_ref.filename
                output_container_id = file_ref.container_id
                print(f"Found generated file: {output_filename} (ID: {output_file_id})")

# If we found a file, download it to the current directory
if output_file_id and output_filename:
    # Retrieve the raw file bytes from the container
    raw_content = chat_client.containers.files.content.retrieve(
        file_id=output_file_id,
        container_id=output_container_id,
    )

    # Write the bytes to a local file
    with open(output_filename, "wb") as local_file:
        local_file.write(raw_content.read())
        print(f"Downloaded: {output_filename}")

    print(f"Chart saved and ready to view: {output_filename}")
else:
    # No file was found in the response
    print("The agent did not generate a downloadable file this time.")